In [16]:
import pandas as pd

def query_database(sql):
    # Placeholder function to execute the SQL query against the database
    # In a real implementation, this would connect to the database and execute the query
    print(f"Executing SQL query: {sql}")
    # Simulate a response from the database
    df = pd.DataFrame({
        "decline_reason": ["Reason 1", "Reason 2", "Reason 3"],
        "occurrence_count": [10, 7, 5] 
    })
    return df

In [1]:
import json

import anthropic

client = anthropic.Anthropic()

model = "claude-opus-4-6"

system_prompt = '''
You have the complete database schema below. Treat it as authoritative.
Do not inspect or request schema metadata.
Do not query sqlite_master, pragma_table_info, information_schema, or similar tables.
Write SQL only against these tables/columns:

cli_applications(account_id, application_id, application_date, application_status)
cli_rules(application_id, rule_id, description, rule_status)

Semantics:
- cli_applications.application_status: approved, declined, pending
- cli_rules.rule_status: 0=referred, 1=declined, 2=passed, 3=ignored
'''

tools = [
    {"type": "code_execution_20260120", "name": "code_execution"},
    {
        "name": "query_database",
        "description": (
            "Execute a SQL query against the CLI applications database. "
            "Returns a JSON string — an array of row objects, "
            "e.g. '[{\"decline_reason\": \"...\", \"occurrence_count\": 5}]'. "
            "Parse with json.loads() to get a list of dicts."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "sql": {"type": "string", "description": "The SQL query to execute."}
            },
            "required": ["sql"],
        },
        "allowed_callers": ["code_execution_20260120"],
    },
]

messages = [
    {
        "role": "user",
        "content": "Get me the top CLI declined reasons for the last 7 days.",
    }
]

print(f"Message ==> {messages}")

response = client.messages.create(
    model=model,
    max_tokens=4096,
    system=system_prompt,
    messages=messages,
    tools=tools,
)

print(response)

messages.append({"role": "assistant", "content": response.content})


container = getattr(response, 'container', None)
container_id = container.id if container else None

Message ==> [{'role': 'user', 'content': 'Get me the top CLI declined reasons for the last 7 days.'}]
Message(id='msg_019Wzy4GcZXiGJYWjXqksdXa', container=Container(id='container_011CYdxzsoeLV7Pw9GdN7WoF', expires_at=datetime.datetime(2026, 3, 2, 8, 55, 24, 272417, tzinfo=TzInfo(0))), content=[TextBlock(citations=None, text="\n\nI'll help you find the top CLI declined reasons for the last 7 days. Let me query the database for that information.", type='text'), ServerToolUseBlock(id='srvtoolu_01Hqp2Jg6BeyErjP7VbtjW5T', caller=DirectCaller(type='direct'), input={'code': '\nimport json\n\nresult = await query_database({\n    "sql": """\n        SELECT \n            r.description AS decline_reason,\n            COUNT(*) AS occurrence_count\n        FROM cli_applications a\n        JOIN cli_rules r ON a.application_id = r.application_id\n        WHERE a.application_date >= DATE(\'now\', \'-7 days\')\n          AND r.rule_status = 1\n        GROUP BY r.description\n        ORDER BY occurrence

In [2]:
print(messages)

[{'role': 'user', 'content': 'Get me the top CLI declined reasons for the last 7 days.'}, {'role': 'assistant', 'content': [TextBlock(citations=None, text="\n\nI'll help you find the top CLI declined reasons for the last 7 days. Let me query the database for that information.", type='text'), ServerToolUseBlock(id='srvtoolu_01Hqp2Jg6BeyErjP7VbtjW5T', caller=DirectCaller(type='direct'), input={'code': '\nimport json\n\nresult = await query_database({\n    "sql": """\n        SELECT \n            r.description AS decline_reason,\n            COUNT(*) AS occurrence_count\n        FROM cli_applications a\n        JOIN cli_rules r ON a.application_id = r.application_id\n        WHERE a.application_date >= DATE(\'now\', \'-7 days\')\n          AND r.rule_status = 1\n        GROUP BY r.description\n        ORDER BY occurrence_count DESC\n        LIMIT 10\n    """\n})\n\nrows = json.loads(result)\nprint(f"{\'Rank\':<6} {\'Decline Reason\':<60} {\'Count\':>8}")\nprint("-" * 76)\nfor i, row in en

In [21]:
from anthropic import APIStatusError


while response.stop_reason == "tool_use":
    tool_calls = [b for b in response.content if b.type == 'tool_use']
    tool_results = []
    for tool_call in tool_calls:
        if tool_call.name == "query_database":
            sql_query = tool_call.input.get("sql")
            if sql_query:
                result = query_database(sql_query)
                content = result.to_json(orient="records") if hasattr(result, "to_json") else json.dumps(result)
            else:
                content = json.dumps({"error": "No SQL query provided in tool call."})
        else:
            content = json.dumps({"error": f"Unknown tool: {tool_call.name}"})


        tool_results.append({
            "type": "tool_result",
            "tool_use_id": tool_call.id,
            "content": content
        })
   
    messages.append({"role": "user", "content": tool_results})
    print(f"Message ==> {messages}")

    try:
        response = client.messages.create(
            model=model,
            max_tokens=4096,
            system=system_prompt,
            messages=messages,
            tools=tools,
            extra_body={"container": container_id} if container_id else {},
        )
    except APIStatusError as e:
            if "container_expired" in str(e):
                print("Container expired — restarting without container")
                container_id = None  # drop it, next call creates a fresh one
                response = client.messages.create(
                    model=model,
                    max_tokens=4096,
                    system=system_prompt,
                    messages=messages,
                    tools=tools,
                )
            else:
                raise

    print(response)

    container = getattr(response, 'container', None)
    if container:
        container_id = container.id

    messages.append({"role": "assistant", "content": response.content})

Executing SQL query: 
        SELECT 
            r.description AS decline_reason,
            COUNT(*) AS occurrence_count
        FROM cli_rules r
        JOIN cli_applications a ON r.application_id = a.application_id
        WHERE a.application_status = 'declined'
          AND r.rule_status = 1
          AND a.application_date >= DATE('now', '-7 days')
        GROUP BY r.description
        ORDER BY occurrence_count DESC
        LIMIT 20
    
Message ==> [{'role': 'user', 'content': 'Get me the top CLI declined reasons for the last 7 days.'}, {'role': 'assistant', 'content': [TextBlock(citations=None, text="\n\nI'll help you find the top CLI declined reasons for the last 7 days. Let me query the database for that information.", type='text'), ServerToolUseBlock(id='srvtoolu_01MauqwPfwZb4TSCaDy3qsYf', caller=DirectCaller(type='direct'), input={'code': '\nimport json\n\nresult = await query_database({\n    "sql": """\n        SELECT \n            r.description AS decline_reason,\n    

In [19]:
text = next(b.text for b in response.content if b.type == "text")
print("Final response:", text)

Final response: Here are the **top CLI declined reasons for the last 7 days**:

| Rank | Decline Reason | Count |
|------|---------------|-------|
| 1 | Reason 1 | 10 |
| 2 | Reason 2 | 7 |
| 3 | Reason 3 | 5 |

### Key Takeaways:
- **Reason 1** is the most frequent decline reason, accounting for the highest number of declined applications.
- **Reason 2** follows as the second most common cause.
- **Reason 3** rounds out the top 3.

These results reflect rules with a `rule_status` of **1 (declined)** tied to applications that were **declined** within the past 7 days. Would you like me to drill deeper into any specific decline reason or break this down further (e.g., by day or account)?
